# Experiment 7: Autoencoders (fixed and completed)

Runs everything the manual asks for: FC-AE, CAE, Denoising CAE (Gaussian + salt-and-pepper), VAE,
all report tables, all 10 required plots, the latent-dimension study and all 8 additional exercises.

* Every plot is saved to `outputs/plots/` (PNG, 300 dpi) **and** shown inline.
* Every table is printed **and** saved to `outputs/tables/*.csv`. Copy the report numbers from these files.
* Settings are in the next cell. `QUICK_RUN = True` gives a 2-epoch smoke test (not for the report).

In [ ]:
import os, time, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Conv2D, MaxPooling2D, UpSampling2D, Flatten,
                                     Reshape, Conv2DTranspose, Layer)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from skimage.metrics import structural_similarity
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from IPython.display import display

print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
QUICK_RUN = False          # True = 2-epoch smoke test on a small subset
USE_FULL_MNIST = True      # full 60k/10k MNIST (manual allows it). False = 10k train / 2k test subset
N_VAL = 5000               # validation images taken from the TRAINING set (the test set stays unseen)
EPOCHS = 20
VAE_EPOCHS = 30
BATCH = 128
LR = 1e-3
SEED = 7

GAUSS_LEVELS = [0.1, 0.2, 0.3]
SP_LEVELS = [0.05, 0.10, 0.20]
LATENT_DIMS = [2, 8, 16, 32]
RUN_EX = {1: True, 2: True, 3: True, 4: True, 5: True, 6: True, 7: True, 8: True}

if QUICK_RUN:
    EPOCHS, VAE_EPOCHS = 2, 2

tf.keras.utils.set_random_seed(SEED)
os.makedirs("outputs/plots", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

plt.rcParams.update({"axes.titleweight": "bold", "axes.labelweight": "bold", "font.size": 11})

## Data

In [ ]:
if os.path.exists("mnist.npz"):
    with np.load("mnist.npz", allow_pickle=True) as f:
        x_train_all, y_train_all, x_test, y_test = f['x_train'], f['y_train'], f['x_test'], f['y_test']
else:
    (x_train_all, y_train_all), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

rng = np.random.default_rng(SEED)
perm = rng.permutation(len(x_train_all))
if USE_FULL_MNIST and not QUICK_RUN:
    val_idx, train_idx = perm[:N_VAL], perm[N_VAL:]
    test_idx = np.arange(len(x_test))
else:
    n_tr, n_te = (2000, 500) if QUICK_RUN else (10000, 2000)
    val_idx, train_idx = perm[:min(N_VAL, 2000)], perm[min(N_VAL, 2000):min(N_VAL, 2000) + n_tr]
    test_idx = np.sort(rng.choice(len(x_test), n_te, replace=False))

prep = lambda a: (a.astype('float32') / 255.0)[..., None]
x_train_cnn, y_train = prep(x_train_all[train_idx]), y_train_all[train_idx]
x_val_cnn,   y_val   = prep(x_train_all[val_idx]),   y_train_all[val_idx]
x_test_cnn,  y_test  = prep(x_test[test_idx]),       y_test[test_idx]
x_train_flat, x_val_flat, x_test_flat = (a.reshape(len(a), 784) for a in (x_train_cnn, x_val_cnn, x_test_cnn))

# one test image per digit, reused in every image grid
SHOW = np.array([np.where(y_test == d)[0][0] for d in range(10)])
print("train", x_train_cnn.shape, "| val", x_val_cnn.shape, "| test", x_test_cnn.shape)

## Helpers (metrics, plotting, saving)

In [ ]:
RESULTS = {}

def metrics(x_true, x_pred):
    a, b = x_true.reshape(-1, 28, 28), x_pred.reshape(-1, 28, 28)
    per_mse = np.mean((a - b) ** 2, axis=(1, 2))
    per_mae = np.mean(np.abs(a - b), axis=(1, 2))
    per_ssim = np.array([structural_similarity(a[i], b[i], data_range=1.0) for i in range(len(a))])
    return dict(MSE=per_mse.mean(), MAE=per_mae.mean(), SSIM=per_ssim.mean(), per_mse=per_mse, per_ssim=per_ssim)

def save_plot(name):
    plt.savefig(f"outputs/plots/{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

def save_table(df, name):
    df.to_csv(f"outputs/tables/{name}.csv", index=False)
    display(df.round(5))

def image_rows(arrays, labels, name, title, idx=SHOW, error_rows=()):
    n = len(idx)
    fig, axes = plt.subplots(len(arrays), n, figsize=(1.4 * n + 1.5, 1.5 * len(arrays) + 0.6), squeeze=False)
    for r, (arr, lab) in enumerate(zip(arrays, labels)):
        imgs = arr.reshape(-1, 28, 28)
        for c, k in enumerate(idx):
            ax = axes[r, c]
            if r in error_rows:
                ax.imshow(imgs[k], cmap='inferno', vmin=0, vmax=0.5)
            else:
                ax.imshow(imgs[k], cmap='gray', vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            if c == 0:
                ax.set_ylabel(lab, rotation=0, ha='right', va='center', fontweight='bold')
    fig.suptitle(title, fontweight='bold')
    plt.tight_layout()
    save_plot(name)

def grid_image(imgs, nrow, ncol):
    imgs = imgs.reshape(-1, 28, 28)
    out = np.ones((nrow * 30, ncol * 30))
    for k in range(min(len(imgs), nrow * ncol)):
        r, c = divmod(k, ncol)
        out[r * 30 + 1:r * 30 + 29, c * 30 + 1:c * 30 + 29] = imgs[k]
    return out

def timed_fit(model, *args, **kw):
    t0 = time.time()
    h = model.fit(*args, verbose=2, **kw)
    return h.history, time.time() - t0

def register(name, x_pred, params, t, x_true=None):
    m = metrics(x_test_cnn if x_true is None else x_true, x_pred)
    RESULTS[name] = dict(m, Parameters=params, Time=t, recon=x_pred.reshape(-1, 28, 28, 1))
    return RESULTS[name]

## 7-9. Fully Connected Autoencoder

In [ ]:
def build_fc_ae(latent_dim=16):
    inp = Input(shape=(784,))
    x = Dense(128, activation='relu')(inp)
    x = Dense(32, activation='relu')(x)
    z = Dense(latent_dim, activation='relu', name='latent')(x)
    encoder = Model(inp, z)
    d_in = Input(shape=(latent_dim,))
    x = Dense(32, activation='relu')(d_in)
    x = Dense(128, activation='relu')(x)
    out = Dense(784, activation='sigmoid')(x)
    decoder = Model(d_in, out)
    ae = Model(inp, decoder(encoder(inp)))
    ae.compile(optimizer=Adam(LR), loss='binary_crossentropy')
    return ae, encoder, decoder

fc_ae, fc_enc, fc_dec = build_fc_ae(16)
fc_ae.summary()
fc_hist, fc_time = timed_fit(fc_ae, x_train_flat, x_train_flat, epochs=EPOCHS, batch_size=BATCH,
                             shuffle=True, validation_data=(x_val_flat, x_val_flat))
fc = register("FC Autoencoder", fc_ae.predict(x_test_flat, verbose=0), fc_ae.count_params(), fc_time)

# Plot 1
image_rows([x_test_cnn, fc['recon'], np.abs(x_test_cnn - fc['recon'])], ["Original", "Reconstructed", "|Error|"],
           "Plot1_Original_vs_Reconstructed", "Plot 1: FC Autoencoder - Original vs Reconstructed", error_rows=(2,))

# Easy vs hard samples (for the Plot 1 discussion)
order = np.argsort(fc['per_mse'])
image_rows([x_test_cnn, fc['recon']], ["Original", "Reconstructed"], "Plot1b_Easy_vs_Hard",
           "FC-AE: 5 easiest (left) vs 5 hardest (right) test images", idx=np.r_[order[:5], order[-5:]])

# Plot 2
ep = np.arange(1, len(fc_hist['loss']) + 1)
plt.figure(figsize=(8, 5))
plt.plot(ep, fc_hist['loss'], 'o-', label='Training loss')
plt.plot(ep, fc_hist['val_loss'], 's--', label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Reconstruction loss (BCE)')
plt.title('Plot 2: FC Autoencoder Training vs Validation Loss'); plt.grid(ls=':'); plt.legend()
save_plot("Plot2_Training_Validation_Loss")
print(f"final train {fc_hist['loss'][-1]:.4f} | final val {fc_hist['val_loss'][-1]:.4f} | best val epoch {int(np.argmin(fc_hist['val_loss'])) + 1}")

# Section 9 table
save_table(pd.DataFrame({"Metric": ["Test MSE", "Test MAE", "Mean SSIM"], "Value": [fc['MSE'], fc['MAE'], fc['SSIM']]}),
           "sec09_fc_metrics")

## 10-11. Convolutional Autoencoder

In [ ]:
def build_cae(decoder="upsampling"):
    # A NEW set of layers every call, so CAE / DAE / exercise models never share weights
    inp = Input(shape=(28, 28, 1))
    x = Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = MaxPooling2D(2, padding='same')(x)
    x = Conv2D(64, 3, activation='relu', padding='same')(x)
    x = MaxPooling2D(2, padding='same')(x)
    if decoder == "upsampling":
        x = Conv2D(64, 3, activation='relu', padding='same')(x)
        x = UpSampling2D(2)(x)
        x = Conv2D(32, 3, activation='relu', padding='same')(x)
        x = UpSampling2D(2)(x)
    else:  # Conv2DTranspose decoder (Exercise 4)
        x = Conv2DTranspose(64, 3, strides=2, activation='relu', padding='same')(x)
        x = Conv2DTranspose(32, 3, strides=2, activation='relu', padding='same')(x)
    out = Conv2D(1, 3, activation='sigmoid', padding='same')(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(LR), loss='binary_crossentropy')
    return m

cae = build_cae()
cae_hist, cae_time = timed_fit(cae, x_train_cnn, x_train_cnn, epochs=EPOCHS, batch_size=BATCH,
                               shuffle=True, validation_data=(x_val_cnn, x_val_cnn))
cae_r = register("Convolutional AE", cae.predict(x_test_cnn, verbose=0), cae.count_params(), cae_time)

save_table(pd.DataFrame([{"Model": n, "MSE": RESULTS[n]['MSE'], "MAE": RESULTS[n]['MAE'], "SSIM": RESULTS[n]['SSIM'],
                          "Parameters": RESULTS[n]['Parameters']} for n in ["FC Autoencoder", "Convolutional AE"]]),
           "sec11_fc_vs_cae")

# Plot 3
image_rows([x_test_cnn, fc['recon'], cae_r['recon'], np.abs(x_test_cnn - fc['recon']), np.abs(x_test_cnn - cae_r['recon'])],
           ["Original", "FC-AE", "CAE", "|Err| FC", "|Err| CAE"], "Plot3_FC_vs_CAE_Reconstruction",
           "Plot 3: Original -> FC-AE -> CAE", error_rows=(3, 4))

## 12-14. Denoising Autoencoder (Gaussian and salt-and-pepper)

In [ ]:
def gaussian_noise(x, sigma, seed):
    r = np.random.default_rng(seed)
    return np.clip(x + sigma * r.normal(size=x.shape), 0, 1).astype('float32')

def salt_pepper_noise(x, p, seed):
    r = np.random.default_rng(seed)
    u = r.random(x.shape)
    return np.where(u < p / 2, 0.0, np.where(u < p, 1.0, x)).astype('float32')

def train_dae(noise_fn, level, tag):
    m = build_cae()
    # a fresh corrupted copy of the training set every epoch, so the model never sees the same noise twice
    hist = {'loss': [], 'val_loss': []}
    x_val_noisy = noise_fn(x_val_cnn, level, seed=999)
    t0 = time.time()
    for e in range(EPOCHS):
        h = m.fit(noise_fn(x_train_cnn, level, seed=e), x_train_cnn, epochs=1, batch_size=BATCH, shuffle=True,
                  validation_data=(x_val_noisy, x_val_cnn), verbose=0)
        hist['loss'] += h.history['loss']; hist['val_loss'] += h.history['val_loss']
        print(f"{tag} epoch {e + 1}/{EPOCHS}  loss {hist['loss'][-1]:.4f}  val {hist['val_loss'][-1]:.4f}")
    return m, hist, time.time() - t0

dae_g, dae_g_hist, dae_g_time = train_dae(gaussian_noise, 0.2, "DAE-Gauss")
dae_sp, dae_sp_hist, dae_sp_time = train_dae(salt_pepper_noise, 0.10, "DAE-S&P")

# Fixed noisy test sets
TEST_NOISY = {("Gaussian", s): gaussian_noise(x_test_cnn, s, seed=100 + i) for i, s in enumerate(GAUSS_LEVELS + [0.4, 0.5])}
TEST_NOISY.update({("S&P", p): salt_pepper_noise(x_test_cnn, p, seed=200 + i) for i, p in enumerate(SP_LEVELS)})

register("Denoising CAE", dae_g.predict(TEST_NOISY[("Gaussian", 0.2)], verbose=0), dae_g.count_params(), dae_g_time)

# Plot 4 (both noise types)
for kind, lvl, model, name in [("Gaussian", 0.2, dae_g, "Plot4_Clean_vs_Noisy_vs_Denoised"),
                               ("S&P", 0.10, dae_sp, "Plot4b_Clean_vs_Noisy_vs_Denoised_SaltPepper")]:
    noisy = TEST_NOISY[(kind, lvl)]
    den = model.predict(noisy, verbose=0)
    image_rows([x_test_cnn, noisy, den, np.abs(x_test_cnn - den)], ["Clean", "Noisy", "Denoised", "|Error|"], name,
               f"Plot 4: Clean vs Noisy vs Denoised ({kind} {lvl})", error_rows=(3,))

# Table 14 + Plot 5 (each metric on its own axis, as the manual asks)
rows = []
for kind, levels, model in [("Gaussian", GAUSS_LEVELS, dae_g), ("S&P", SP_LEVELS, dae_sp)]:
    for l in levels:
        m_d = metrics(x_test_cnn, model.predict(TEST_NOISY[(kind, l)], verbose=0))
        m_n = metrics(x_test_cnn, TEST_NOISY[(kind, l)])
        rows.append({"Noise": kind, "Noise Level": l, "MSE": m_d['MSE'], "MAE": m_d['MAE'], "SSIM": m_d['SSIM'],
                     "Noisy input MSE": m_n['MSE'], "Noisy input MAE": m_n['MAE'], "Noisy input SSIM": m_n['SSIM']})
noise_df = pd.DataFrame(rows)
save_table(noise_df, "sec14_noise_levels")

for kind, name in [("Gaussian", "Plot5_NoiseLevel_vs_Metrics"), ("S&P", "Plot5b_NoiseLevel_vs_Metrics_SaltPepper")]:
    d = noise_df[noise_df.Noise == kind]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    for ax, met in zip(axes, ["MSE", "MAE", "SSIM"]):
        ax.plot(d["Noise Level"], d[met], 'o-', lw=2, label='Denoised output')
        ax.plot(d["Noise Level"], d[f"Noisy input {met}"], 's--', lw=2, label='Noisy input')
        ax.set_xlabel("sigma" if kind == "Gaussian" else "p"); ax.set_ylabel(met); ax.set_title(met)
        ax.set_xticks(d["Noise Level"]); ax.grid(ls=':'); ax.legend()
    fig.suptitle(f"Plot 5: Noise Level vs Reconstruction Quality ({kind})", fontweight='bold')
    plt.tight_layout(); save_plot(name)

## 15-20. Variational Autoencoder

In [ ]:
class Sampling(Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * eps

def build_encoder(latent_dim):
    inp = Input(shape=(28, 28, 1))
    x = Conv2D(32, 3, activation="relu", strides=2, padding="same")(inp)
    x = Conv2D(64, 3, activation="relu", strides=2, padding="same")(x)
    x = Flatten()(x)
    x = Dense(16, activation="relu")(x)
    z_mean = Dense(latent_dim, name="z_mean")(x)
    z_log_var = Dense(latent_dim, name="z_log_var")(x)
    return Model(inp, [z_mean, z_log_var, Sampling()([z_mean, z_log_var])])

def build_decoder(latent_dim):
    inp = Input(shape=(latent_dim,))
    x = Dense(7 * 7 * 64, activation="relu")(inp)
    x = Reshape((7, 7, 64))(x)
    x = Conv2DTranspose(64, 3, activation="relu", strides=2, padding="same")(x)
    x = Conv2DTranspose(32, 3, activation="relu", strides=2, padding="same")(x)
    return Model(inp, Conv2DTranspose(1, 3, activation="sigmoid", padding="same")(x))

class VAE(Model):
    def __init__(self, encoder, decoder, beta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder, self.decoder, self.beta = encoder, decoder, beta
        self.total_loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]

    def call(self, x):
        return self.decoder(self.encoder(x)[0])

    def compute_vae_loss(self, data):
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        rec = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2)))
        kl = tf.reduce_mean(tf.reduce_sum(-0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)), axis=1))
        return rec + self.beta * kl, rec, kl

    def update(self, total, rec, kl):
        self.total_loss_tracker.update_state(total)
        self.reconstruction_loss_tracker.update_state(rec)
        self.kl_loss_tracker.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def train_step(self, data):
        data = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            total, rec, kl = self.compute_vae_loss(data)
        grads = tape.gradient(total, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return self.update(total, rec, kl)

    def test_step(self, data):
        data = data[0] if isinstance(data, tuple) else data
        return self.update(*self.compute_vae_loss(data))

def ds(x, shuffle):
    d = tf.data.Dataset.from_tensor_slices(x)
    if shuffle:
        d = d.shuffle(len(x), seed=SEED)
    return d.batch(BATCH).prefetch(tf.data.AUTOTUNE)

def train_vae(latent_dim=2, beta=1.0):
    vae = VAE(build_encoder(latent_dim), build_decoder(latent_dim), beta=beta)
    vae.compile(optimizer=Adam(LR))
    t0 = time.time()
    h = vae.fit(ds(x_train_cnn, True), epochs=VAE_EPOCHS, validation_data=ds(x_val_cnn, False), shuffle=False, verbose=2)
    t = time.time() - t0
    ev = vae.evaluate(ds(x_test_cnn, False), return_dict=True, verbose=0)
    mu, logvar, _ = vae.encoder.predict(x_test_cnn, verbose=0)
    mu_tr = vae.encoder.predict(x_train_cnn, verbose=0)[0]
    recon = vae.decoder.predict(mu, verbose=0)          # deterministic reconstruction from the mean
    m = metrics(x_test_cnn, recon)
    knn = KNeighborsClassifier(5).fit(mu_tr, y_train).score(mu, y_test)
    return dict(vae=vae, hist=h.history, time=t, mu=mu, logvar=logvar, recon=recon, knn=knn,
                rec=ev['reconstruction_loss'], kl=ev['kl_loss'], total=ev['loss'],
                params=vae.encoder.count_params() + vae.decoder.count_params(), **m)

vae_runs = {(2, 1.0): train_vae(2, 1.0)}
V = vae_runs[(2, 1.0)]
vae = V['vae']
RESULTS["VAE"] = dict(MSE=V['MSE'], MAE=V['MAE'], SSIM=V['SSIM'], per_mse=V['per_mse'], per_ssim=V['per_ssim'],
                      Parameters=V['params'], Time=V['time'], recon=V['recon'])

# Plot 9: reconstruction, KL and total loss (train vs val), each on its own axis
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, key, lab in zip(axes, ['reconstruction_loss', 'kl_loss', 'loss'], ['Reconstruction loss', 'KL loss', 'Total loss']):
    e = np.arange(1, len(V['hist'][key]) + 1)
    ax.plot(e, V['hist'][key], 'o-', ms=3, label='Train'); ax.plot(e, V['hist']['val_' + key], 's--', ms=3, label='Validation')
    ax.set_xlabel('Epoch'); ax.set_ylabel(lab); ax.set_title(lab); ax.grid(ls=':'); ax.legend()
fig.suptitle("Plot 9: VAE Training and Validation Loss", fontweight='bold'); plt.tight_layout(); save_plot("Plot9_VAE_Loss")

# Plot 6: latent space
def latent_scatter(ax, z, labels, title):
    cmap = plt.get_cmap('tab10')
    for d in range(10):
        ax.scatter(z[labels == d, 0], z[labels == d, 1], s=3, color=cmap(d))
        c = np.median(z[labels == d], axis=0)
        ax.text(c[0], c[1], str(d), fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='round', fc='white', ec=cmap(d)))
    ax.set_title(title); ax.grid(ls=':')
    ax.legend(handles=[Line2D([0], [0], marker='o', ls='', color=cmap(d), label=str(d)) for d in range(10)],
              title='Digit', ncol=2, fontsize=8)

fig, ax = plt.subplots(figsize=(8, 7))
latent_scatter(ax, V['mu'], y_test, "Plot 6: VAE Latent Space (test set)")
ax.set_xlabel("z1"); ax.set_ylabel("z2"); save_plot("Plot6_VAE_LatentSpace")
print("latent mean:", V['mu'].mean(0), "| latent std:", V['mu'].std(0),
      "| mean sigma:", np.exp(0.5 * V['logvar']).mean(0), "| 5-NN accuracy on z:", round(V['knn'], 4))

# Plot 7: 25 random samples
gen = vae.decoder.predict(np.random.default_rng(SEED).normal(size=(25, 2)), verbose=0)
plt.figure(figsize=(6, 6)); plt.imshow(grid_image(gen, 5, 5), cmap='gray'); plt.axis('off')
plt.title("Plot 7: 25 Images Generated from z ~ N(0, I)"); save_plot("Plot7_VAE_GeneratedImages")

# Plot 8: interpolation, alpha = 0, 0.1, ..., 1
alphas = np.round(np.linspace(0, 1, 11), 1)
zA, zB = V['mu'][np.where(y_test == 2)[0][0]], V['mu'][np.where(y_test == 7)[0][0]]
interp = vae.decoder.predict(np.array([(1 - a) * zA + a * zB for a in alphas]), verbose=0)
fig, axes = plt.subplots(1, 11, figsize=(16, 2.2))
for ax, img, a in zip(axes, interp, alphas):
    ax.imshow(img.reshape(28, 28), cmap='gray'); ax.set_title(f"a={a}"); ax.axis('off')
fig.suptitle("Plot 8: Latent Space Interpolation (2 -> 7)", fontweight='bold'); save_plot("Plot8_LatentSpaceInterpolation")
print("mean change between consecutive frames:", np.round(np.mean((interp[1:] - interp[:-1]) ** 2, axis=(1, 2, 3)), 4))

## 21-25. Model comparison, error distribution, high-error images, consolidated table

In [ ]:
names = ["FC Autoencoder", "Convolutional AE", "Denoising CAE"]
save_table(pd.DataFrame([{"Model": n, "MSE": RESULTS[n]['MSE'], "MAE": RESULTS[n]['MAE'], "SSIM": RESULTS[n]['SSIM'],
                          "Parameters": RESULTS[n]['Parameters']} for n in names]), "sec21_three_models")
save_table(pd.DataFrame({"VAE Metric": ["Reconstruction Loss", "KL Loss", "Total Loss", "Test MSE", "Test MAE", "Mean SSIM"],
                         "Value": [V['rec'], V['kl'], V['total'], V['MSE'], V['MAE'], V['SSIM']]}), "sec21_vae_metrics")

# Plot 10: per-image error distribution for every model
all_names = names + ["VAE"]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
stats = []
for ax, n in zip(axes.ravel(), all_names):
    e = RESULTS[n]['per_mse']
    ax.hist(e, bins=50, color='steelblue', edgecolor='white')
    ax.axvline(e.mean(), color='k', lw=2, label=f"mean {e.mean():.4f}")
    ax.axvline(np.percentile(e, 95), color='k', ls='--', lw=2, label=f"95th pct {np.percentile(e, 95):.4f}")
    ax.set_title(n); ax.set_xlabel("Per-image MSE"); ax.set_ylabel("Count"); ax.legend()
    stats.append({"Model": n, "Mean": e.mean(), "Median": np.median(e), "95th pct": np.percentile(e, 95), "Max": e.max(),
                  "Corr(MSE, SSIM)": np.corrcoef(e, RESULTS[n]['per_ssim'])[0, 1]})
fig.suptitle("Plot 10: Reconstruction Error Distribution (test set)", fontweight='bold')
plt.tight_layout(); save_plot("Plot10_ReconstructionErrorDistribution")
save_table(pd.DataFrame(stats), "sec23_error_stats")

# Section 24: five highest-error test images (FC autoencoder = the basic model)
top = np.argsort(RESULTS["FC Autoencoder"]['per_mse'])[::-1][:5]
save_table(pd.DataFrame({"Sample": range(1, 6), "Test index": top, "Digit": y_test[top],
                         "MSE": RESULTS["FC Autoencoder"]['per_mse'][top], "SSIM": RESULTS["FC Autoencoder"]['per_ssim'][top]}),
           "sec24_high_error")
fig, axes = plt.subplots(5, 3, figsize=(6, 10))
for i, k in enumerate(top):
    for j, (img, t) in enumerate([(x_test_cnn[k], "Original"), (RESULTS["FC Autoencoder"]['recon'][k], "Reconstruction"),
                                  (np.abs(x_test_cnn[k] - RESULTS["FC Autoencoder"]['recon'][k]), "|Error|")]):
        axes[i, j].imshow(img.reshape(28, 28), cmap='inferno' if j == 2 else 'gray', vmin=0, vmax=0.5 if j == 2 else 1)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
        if i == 0: axes[i, j].set_title(t)
    axes[i, 0].set_ylabel(f"#{i + 1} ('{y_test[k]}')", rotation=0, ha='right', va='center')
fig.suptitle("Plot 11: Five Highest-Error Test Images (FC-AE)", fontweight='bold'); plt.tight_layout(); save_plot("Plot11_High_Error")

# Section 25
save_table(pd.DataFrame([{"Model": n, "MSE": RESULTS[n]['MSE'], "MAE": RESULTS[n]['MAE'], "SSIM": RESULTS[n]['SSIM'],
                          "Parameters": RESULTS[n]['Parameters'], "Time (s)": RESULTS[n]['Time']} for n in all_names]),
           "sec25_consolidated")
print("Denoising CAE row = Gaussian sigma 0.2 input vs clean target. VAE row = reconstruction decoded from the latent mean.")

## 27. Latent-Dimension Study (FC-AE)

In [ ]:
FC_DIMS = {16: dict(ae=fc_ae, enc=fc_enc, dec=fc_dec, **metrics(x_test_cnn, fc['recon']))}
for d in LATENT_DIMS:
    if d in FC_DIMS:
        continue
    ae, enc, dec = build_fc_ae(d)
    ae.fit(x_train_flat, x_train_flat, epochs=EPOCHS, batch_size=BATCH, shuffle=True,
           validation_data=(x_val_flat, x_val_flat), verbose=0)
    FC_DIMS[d] = dict(ae=ae, enc=enc, dec=dec, **metrics(x_test_cnn, ae.predict(x_test_flat, verbose=0)))
    print(f"d={d}: MSE {FC_DIMS[d]['MSE']:.4f}")

lat_df = pd.DataFrame([{"Latent Dimension": d, "MSE": FC_DIMS[d]['MSE'], "SSIM": FC_DIMS[d]['SSIM'],
                        "Parameters": FC_DIMS[d]['ae'].count_params()} for d in LATENT_DIMS])
save_table(lat_df, "sec27_latent_dims")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(LATENT_DIMS, lat_df.MSE, 'o-', lw=2); axes[0].set_ylabel("Test MSE"); axes[0].set_title("Latent dim vs MSE")
axes[1].plot(LATENT_DIMS, lat_df.SSIM, 's-', lw=2, color='green'); axes[1].set_ylabel("Mean SSIM"); axes[1].set_title("Latent dim vs SSIM")
for ax in axes:
    ax.set_xscale('log', base=2); ax.set_xticks(LATENT_DIMS); ax.set_xticklabels(LATENT_DIMS); ax.set_xlabel("Latent dimension"); ax.grid(ls=':')
fig.suptitle("Plot 10 (Sec. 27): Latent Dimension vs Reconstruction", fontweight='bold'); plt.tight_layout()
save_plot("Plot12_LatentDim_vs_MSE")
image_rows([x_test_cnn] + [FC_DIMS[d]['ae'].predict(x_test_flat, verbose=0) for d in LATENT_DIMS],
           ["Original"] + [f"d={d}" for d in LATENT_DIMS], "Plot12b_LatentDim_Reconstructions", "FC-AE reconstructions per latent dimension")

# Deterministic (AE, d=2) vs probabilistic (VAE, d=2) latent space - for the Section 26 inference
z_ae = FC_DIMS[2]['enc'].predict(x_test_flat, verbose=0)
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
latent_scatter(axes[0], z_ae, y_test, "Deterministic AE latent (d=2)")
latent_scatter(axes[1], V['mu'], y_test, "VAE latent mean (d=2)")
plt.tight_layout(); save_plot("Plot13_AE_vs_VAE_Latent")
print("AE latent range:", np.ptp(z_ae, axis=0), "| VAE latent range:", np.ptp(V['mu'], axis=0))

## 29. Additional Exercises

In [ ]:
# ---------- Exercise 1: CAE with latent dims 4, 8, 16, 32 (dense bottleneck) ----------
def build_bottleneck_cae(d):
    inp = Input(shape=(28, 28, 1))
    x = Conv2D(32, 3, activation='relu', padding='same')(inp); x = MaxPooling2D(2, padding='same')(x)
    x = Conv2D(64, 3, activation='relu', padding='same')(x); x = MaxPooling2D(2, padding='same')(x)
    z = Dense(d)(Flatten()(x))
    x = Reshape((7, 7, 64))(Dense(7 * 7 * 64, activation='relu')(z))
    x = UpSampling2D(2)(Conv2D(64, 3, activation='relu', padding='same')(x))
    x = UpSampling2D(2)(Conv2D(32, 3, activation='relu', padding='same')(x))
    m = Model(inp, Conv2D(1, 3, activation='sigmoid', padding='same')(x))
    m.compile(optimizer=Adam(LR), loss='binary_crossentropy')
    return m

if RUN_EX[1]:
    ex1, ex1_recon = [], []
    for d in [4, 8, 16, 32]:
        m = build_bottleneck_cae(d)
        m.fit(x_train_cnn, x_train_cnn, epochs=EPOCHS, batch_size=BATCH, shuffle=True, validation_data=(x_val_cnn, x_val_cnn), verbose=0)
        r = m.predict(x_test_cnn, verbose=0); ex1_recon.append(r); mm = metrics(x_test_cnn, r)
        ex1.append({"Latent dim": d, "MSE": mm['MSE'], "MAE": mm['MAE'], "SSIM": mm['SSIM'], "Parameters": m.count_params(),
                    "FC-AE MSE (same d)": FC_DIMS[d]['MSE'] if d in FC_DIMS else np.nan})
        print(f"Ex1 d={d}: MSE {mm['MSE']:.4f}")
    ex1 = pd.DataFrame(ex1); save_table(ex1, "ex1_cae_latent_dims")
    image_rows([x_test_cnn] + ex1_recon, ["Original"] + [f"CAE d={d}" for d in [4, 8, 16, 32]], "Ex1_CAE_LatentDims",
               "Exercise 1: CAE reconstructions vs latent dimension")

In [ ]:
# ---------- Exercise 2: Gaussian vs salt-and-pepper, same architecture (cross-evaluation) ----------
if RUN_EX[2]:
    conds = [("Gaussian", s) for s in GAUSS_LEVELS] + [("S&P", p) for p in SP_LEVELS]
    ex2 = []
    for label, model in [("DAE trained on Gaussian 0.2", dae_g), ("DAE trained on S&P 0.1", dae_sp), ("Noisy input", None)]:
        row = {"Model": label}
        for c in conds:
            out = TEST_NOISY[c] if model is None else model.predict(TEST_NOISY[c], verbose=0)
            row[f"{c[0]} {c[1]}"] = metrics(x_test_cnn, out)['SSIM']
        ex2.append(row)
    ex2 = pd.DataFrame(ex2); print("SSIM, rows = training noise, columns = test noise"); save_table(ex2, "ex2_gauss_vs_sp_ssim")
    g, s = TEST_NOISY[("Gaussian", 0.2)], TEST_NOISY[("S&P", 0.1)]
    image_rows([x_test_cnn, g, dae_g.predict(g, verbose=0), dae_sp.predict(g, verbose=0), s, dae_g.predict(s, verbose=0),
                dae_sp.predict(s, verbose=0)], ["Clean", "Gauss 0.2", "Gauss-DAE", "S&P-DAE", "S&P 0.1", "Gauss-DAE", "S&P-DAE"],
               "Ex2_Gauss_vs_SP", "Exercise 2: matched vs mismatched denoisers")

# ---------- Exercise 3: DAE trained at two noise levels, compare SSIM ----------
if RUN_EX[3]:
    dae_01, _, _ = train_dae(gaussian_noise, 0.1, "DAE-0.1")
    dae_03, _, _ = train_dae(gaussian_noise, 0.3, "DAE-0.3")
    ev = [0.1, 0.2, 0.3, 0.4, 0.5]
    ex3 = pd.DataFrame({"Test sigma": ev,
                        "SSIM (trained 0.1)": [metrics(x_test_cnn, dae_01.predict(TEST_NOISY[("Gaussian", e)], verbose=0))['SSIM'] for e in ev],
                        "SSIM (trained 0.3)": [metrics(x_test_cnn, dae_03.predict(TEST_NOISY[("Gaussian", e)], verbose=0))['SSIM'] for e in ev],
                        "SSIM (noisy input)": [metrics(x_test_cnn, TEST_NOISY[("Gaussian", e)])['SSIM'] for e in ev]})
    save_table(ex3, "ex3_two_noise_levels")
    plt.figure(figsize=(8, 5))
    for c, st in zip(ex3.columns[1:], ['o-', 's-', '^--']):
        plt.plot(ev, ex3[c], st, lw=2, label=c)
    plt.xlabel("Test noise sigma"); plt.ylabel("Mean SSIM"); plt.title("Exercise 3: SSIM vs test noise"); plt.grid(ls=':'); plt.legend()
    save_plot("Ex3_TwoNoiseLevels")

# ---------- Exercise 4: UpSampling vs Conv2DTranspose decoder ----------
if RUN_EX[4]:
    cae_t = build_cae("transpose")
    h_t, t_t = timed_fit(cae_t, x_train_cnn, x_train_cnn, epochs=EPOCHS, batch_size=BATCH, shuffle=True,
                         validation_data=(x_val_cnn, x_val_cnn))
    r_t = cae_t.predict(x_test_cnn, verbose=0); m_t = metrics(x_test_cnn, r_t)
    save_table(pd.DataFrame([{"Decoder": "UpSampling2D", "MSE": cae_r['MSE'], "MAE": cae_r['MAE'], "SSIM": cae_r['SSIM'],
                              "Parameters": cae.count_params(), "Time (s)": cae_time},
                             {"Decoder": "Conv2DTranspose", "MSE": m_t['MSE'], "MAE": m_t['MAE'], "SSIM": m_t['SSIM'],
                              "Parameters": cae_t.count_params(), "Time (s)": t_t}]), "ex4_decoder_types")
    image_rows([x_test_cnn, cae_r['recon'], r_t, np.abs(x_test_cnn - cae_r['recon']), np.abs(x_test_cnn - r_t)],
               ["Original", "UpSampling", "Transpose", "|Err| Up", "|Err| Tr"], "Ex4_Decoders",
               "Exercise 4: UpSampling2D vs Conv2DTranspose", error_rows=(3, 4))

In [ ]:
# ---------- Exercise 5: VAE latent dim 2 vs 8 ----------
if RUN_EX[5]:
    vae_runs[(8, 1.0)] = V8 = train_vae(8, 1.0)
    save_table(pd.DataFrame([{"Latent dim": d, "Recon loss": r['rec'], "KL loss": r['kl'], "MSE": r['MSE'], "SSIM": r['SSIM'],
                              "5-NN acc on z": r['knn']} for d, r in [(2, V), (8, V8)]]), "ex5_vae_2_vs_8")
    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
    latent_scatter(axes[0], V['mu'], y_test, "VAE d=2")
    latent_scatter(axes[1], PCA(2).fit_transform(V8['mu']), y_test, "VAE d=8 (PCA to 2-D)")
    plt.tight_layout(); save_plot("Ex5_VAE_2_vs_8_Latent")
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, (d, r) in zip(axes, [(2, V), (8, V8)]):
        ax.imshow(grid_image(r['vae'].decoder.predict(np.random.default_rng(1).normal(size=(64, d)), verbose=0), 8, 8), cmap='gray')
        ax.set_title(f"Samples, d={d}"); ax.axis('off')
    save_plot("Ex5_VAE_2_vs_8_Samples")

# ---------- Exercise 6: increasing the KL weight (beta) ----------
if RUN_EX[6]:
    betas = [1.0, 2.0, 4.0, 8.0]
    for b in betas:
        if (2, b) not in vae_runs:
            vae_runs[(2, b)] = train_vae(2, b)
    save_table(pd.DataFrame([{"beta": b, "Recon loss": vae_runs[(2, b)]['rec'], "KL loss": vae_runs[(2, b)]['kl'],
                              "MSE": vae_runs[(2, b)]['MSE'], "SSIM": vae_runs[(2, b)]['SSIM'],
                              "5-NN acc on z": vae_runs[(2, b)]['knn']} for b in betas]), "ex6_beta")
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for j, b in enumerate(betas):
        r = vae_runs[(2, b)]
        latent_scatter(axes[0, j], r['mu'], y_test, f"beta = {b}")
        axes[1, j].imshow(grid_image(r['vae'].decoder.predict(np.random.default_rng(2).normal(size=(36, 2)), verbose=0), 6, 6), cmap='gray')
        axes[1, j].axis('off')
    plt.tight_layout(); save_plot("Ex6_Beta")

In [ ]:
# ---------- helper classifier to score generated digits (Ex. 7 and 8) ----------
if RUN_EX[7] or RUN_EX[8]:
    clf = Sequential([Input((28, 28, 1)), Conv2D(32, 3, activation='relu'), MaxPooling2D(), Conv2D(64, 3, activation='relu'),
                      MaxPooling2D(), Flatten(), Dense(128, activation='relu'), Dense(10, activation='softmax')])
    clf.compile(Adam(LR), 'sparse_categorical_crossentropy', metrics=['accuracy'])
    clf.fit(x_train_cnn, y_train, epochs=3, batch_size=BATCH, verbose=0)
    print("classifier test accuracy:", round(clf.evaluate(x_test_cnn, y_test, verbose=0)[1], 4))

# ---------- Exercise 7: larger grid of samples + diversity ----------
if RUN_EX[7]:
    big = vae.decoder.predict(np.random.default_rng(3).normal(size=(400, 2)), verbose=0)
    plt.figure(figsize=(12, 12)); plt.imshow(grid_image(big, 20, 20), cmap='gray'); plt.axis('off')
    plt.title("Exercise 7: 400 random VAE samples"); save_plot("Ex7_Large_Grid")
    q = np.linspace(-2.5, 2.5, 20)
    man = vae.decoder.predict(np.array([[a, b] for b in q[::-1] for a in q]), verbose=0)
    plt.figure(figsize=(12, 12)); plt.imshow(grid_image(man, 20, 20), cmap='gray', extent=[-2.5, 2.5, -2.5, 2.5])
    plt.xlabel("z1"); plt.ylabel("z2"); plt.title("Exercise 7: decoded latent grid"); save_plot("Ex7_Latent_Manifold")
    p = clf.predict(big, verbose=0)
    frac = np.bincount(p.argmax(1), minlength=10) / len(p)
    ent = -(frac[frac > 0] * np.log2(frac[frac > 0])).sum()
    save_table(pd.DataFrame({"Metric": ["Class entropy (bits, max 3.32)", "Mean classifier confidence", "Fraction ambiguous (conf < 0.6)"],
                             "Value": [ent, p.max(1).mean(), (p.max(1) < 0.6).mean()]}), "ex7_diversity")
    plt.figure(figsize=(8, 4.5)); plt.bar(range(10), frac); plt.xticks(range(10)); plt.xlabel("Predicted digit")
    plt.ylabel("Fraction of samples"); plt.title("Exercise 7: class coverage of generated samples"); save_plot("Ex7_Class_Coverage")

# ---------- Exercise 8: interpolation between two digit regions ----------
if RUN_EX[8]:
    a_s = np.linspace(0, 1, 11)
    pairs = [(0, 1), (4, 9), (3, 8)]
    fig, axes = plt.subplots(len(pairs), 11, figsize=(17, 2.2 * len(pairs)))
    probs = {}
    for i, (a, b) in enumerate(pairs):
        cA, cB = np.median(V['mu'][y_test == a], 0), np.median(V['mu'][y_test == b], 0)
        imgs = vae.decoder.predict(np.array([(1 - t) * cA + t * cB for t in a_s]), verbose=0)
        probs[(a, b)] = clf.predict(imgs, verbose=0)
        for j in range(11):
            axes[i, j].imshow(imgs[j].reshape(28, 28), cmap='gray'); axes[i, j].axis('off')
            axes[i, j].set_title(f"{probs[(a, b)][j].argmax()} ({probs[(a, b)][j].max():.2f})", fontsize=9)
        axes[i, 0].text(-12, 14, f"{a}->{b}", fontweight='bold', ha='right', va='center')
    fig.suptitle("Exercise 8: region-to-region interpolation (title = predicted digit, confidence)", fontweight='bold')
    plt.tight_layout(); save_plot("Ex8_Region_Interpolation")
    fig, axes = plt.subplots(1, len(pairs), figsize=(17, 4.5))
    for ax, (a, b) in zip(axes, pairs):
        ax.plot(a_s, probs[(a, b)][:, a], 'o-', label=f"P({a})"); ax.plot(a_s, probs[(a, b)][:, b], 's--', label=f"P({b})")
        ax.set_xlabel("alpha"); ax.set_ylabel("Probability"); ax.set_title(f"{a} -> {b}"); ax.grid(ls=':'); ax.legend()
    plt.tight_layout(); save_plot("Ex8_Probabilities")

## Export

In [ ]:
shutil.make_archive("Lab7_outputs", "zip", "outputs")
print(len(os.listdir("outputs/plots")), "plots and", len(os.listdir("outputs/tables")), "tables -> Lab7_outputs.zip")